In [ ]:
import tensorflow as tf

from load_data import load_datasets
from hyperparameter_search import CustomHyperModel
from model_construction import model_builder
from helper_functions import eval_model,display_samples
%matplotlib inline

In [ ]:
image_size=512
batch_size=16
dataset_path='130kv2/'
class_map = {0:"Original", 1:"Poisoned"}

In [ ]:
# load our training, validation and test datasets
train_ds, val_ds, test_ds = load_datasets(dataset_path, image_size, batch_size)

# display 10 samples to make sure they loaded OK
display_samples(train_ds,class_map)

# and check how many we have in each dataset
print('Train dataset: ')
print(train_ds.cardinality())
print('\nValidation dataset: ')
print(val_ds.cardinality())
print('\nTest dataset: ')
print(test_ds.cardinality())

# lastly batch them up and cache them
train_ds = train_ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)

In [4]:
import keras_tuner as kt 

# see https://www.tensorflow.org/tutorials/keras/keras_tuner
tuner = kt.Hyperband(
    CustomHyperModel(input_shape=(image_size,image_size) + (3,), num_classes=2),
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='autotune',
    project_name='hyperband'
)

# tuner.search_space_summary()

callbacks = [
    #tf.keras.callbacks.ModelCheckpoint("models/save_at_{epoch}.keras"),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
]

tuner.search(
    train_ds,
    epochs=500,
    callbacks=[callbacks],
    validation_data=val_ds,
)


KeyboardInterrupt: 

In [ ]:
model = model_builder(input_shape=(image_size,image_size) + (3,), num_classes=2)
#tf.keras.utils.plot_model(model, 'models/model.png', show_shapes=True, rankdir='TB')
print(model)

In [ ]:
epochs = 10

callbacks = [
    #tf.keras.callbacks.ModelCheckpoint("models/save_at_{epoch}.keras"),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
]
model_history = model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds,
).history

In [ ]:
#model = tf.keras.models.load_model('save_at_3.keras')
eval_results = eval_model('initial_model',model,model_history,val_ds,test_ds)
#model_history